

# RAG con Ollama y pgvector

### Construyendo memoria semántica para un LLM usando Embeddings y Retrieval-Augmented Generation (RAG)

Notebook desarrollado por [Jhonatan Camasca](https://github.com/jhonatancamasca/) · [Soy Henry](https://www.soyhenry.com/)

---

# Objetivo de la clase

#### En esta clase construiremos un sistema de **Preguntas y Respuestas (Q&A)** basado en la arquitectura **RAG (Retrieval-Augmented Generation)**.

El flujo completo permitirá:

1. Leer información desde un documento PDF.
2. Dividir el contenido en fragmentos de texto (*chunks*).
3. Transformar cada fragmento en un embedding vectorial.
4. Almacenar los embeddings en PostgreSQL usando `pgvector`.
5. Recuperar los fragmentos más relevantes para una consulta.
6. Generar respuestas contextualizadas usando un LLM local con Ollama.


### Fase de Ingesta: Lo que ocurre al leer el documento y guardarlo en la base de datos de vectores.
```
PDF  →  Chunking  →  nomic-embed-text  →  pgvector
```

#### ¿Qué ocurre en esta fase?


| Paso | Acción Realizada | Detalle Técnico |
| :--- | :--- | :--- |
| **1** | Conversión a texto | El PDF se extrae como texto plano (procesando caracteres como `\n`). |
| **2** | *Chunking* | El texto largo se divide en fragmentos pequeños y manejables. |
| **3** | *Embedding* | Cada fragmento se transforma en un vector numérico con `nomic-embed-text`. |
| **4** | Almacenamiento | Los vectores se guardan en PostgreSQL usando la extensión `pgvector`. |
 


### Fase de Inferencia (RAG): Lo que ocurre al hacer una pregunta
```
Pregunta  →  nomic-embed-text  →  pgvector (<=>)  →  top-K chunks  →  llama3.2:1b  →  Respuesta
```

#### ¿Qué ocurre en esta fase?

| Paso | Acción Realizada | Detalle Técnico |
| :--- | :--- | :--- |
| **1** | Consulta del usuario | El usuario realiza una pregunta en lenguaje natural a través de la interfaz. |
| **2** | *Vectorización* | La consulta se transforma en un vector usando el mismo modelo `nomic-embed-text`. |
| **3** | *Búsqueda Semántica* | Se buscan los fragmentos más cercanos en la base de datos `pgvector` usando similitud de coseno. |
| **4** | *Aumento del Prompt* | Se junta la pregunta original con los fragmentos recuperados para dar contexto al modelo. |
| **5** | *Generación (LLM)* | El modelo local en `Ollama` procesa el prompt aumentado y genera la respuesta final. |

---


## Tabla de contenidos
1. [Introducción](#1.-Introduccion)

2. [Stack Tecnológico](#2.-Stack-Tecnologico)
 
3. [Antes de la clase](#3.-Antes-de-la-clase)

4. [Dependencias y configuraciones](#4.-Dependencias-y-configuraciones)

5. [Indexing de texto](#5.-Indexing-de-texto)

6. [Retrieval y búsqueda semántica](6.-Retrieval-y-busqueda-semantica)

7. [Generación de respuestas con RAG](#7.-Generacion-de-respuestas-con-RAG)

8. [Demo de preguntas](#8.-Demo-de-preguntas)






# 1. Introduccion

Aquí la introducción

---

# 2. Stack tecnologico

En este proyecto utilizaremos diferentes herramientas que trabajan juntas para construir un sistema RAG.  
Cada componente cumple una función específica dentro del flujo de preguntas y respuestas.

| Componente | Herramienta | ¿Qué hace? |
|---|---|---|
| Embeddings | `nomic-embed-text` vía Ollama | Convierte textos en vectores numéricos para que la IA pueda entender similitud semántica entre frases y documentos. |
| Modelo LLM | `llama3.1:8b` vía Ollama | Genera respuestas utilizando el contexto recuperado desde la base vectorial. |
| Base vectorial | PostgreSQL + `pgvector` | Almacena embeddings y permite buscar los fragmentos más parecidos a una pregunta. |
| Orquestación | Python + LangChain | Conecta todos los componentes del pipeline RAG. |
| Ejecución local | Ollama | Permite ejecutar modelos de IA de manera local sin depender de APIs externas. |



---


# 3. Antes de la clase

Antes de ejecutar este notebook es necesario preparar el entorno de trabajo.

Existen dos opciones:

1. Configurar manualmente PostgreSQL, pgvector y los modelos de Ollama.
2. Utilizar el entorno Docker proporcionado en el repositorio del proyecto, el cual automatiza toda la configuración.

Se recomienda utilizar la configuración mediante Docker para evitar problemas de compatibilidad y simplificar la instalación.

---

## Opción 1 — Configuración manual

### Levantar PostgreSQL con pgvector

```bash
docker run -d --name pgvector-rag \
  -e POSTGRES_PASSWORD=postgres \
  -e POSTGRES_DB=rag_demo \
  -p 5432:5432 \
  pgvector/pgvector:pg16
```

### Descargar los modelos necesarios en Ollama

```bash
ollama pull nomic-embed-text
ollama pull llama3.1:8b
```

---

## Opción 2 — Configuración automática mediante Docker

El repositorio del proyecto incluye un `docker-compose` y un entorno preconfigurado que automatiza:

- La creación de PostgreSQL con pgvector.
- La configuración de la base de datos.
- La conexión entre servicios.
- La preparación del entorno para ejecutar el notebook.

Para utilizar esta opción, seguir las instrucciones del archivo `README.md` del repositorio.

---

---
# 4. Dependencias y configuraciones

En esta sección prepararemos el entorno de trabajo del notebook.

Primero instalaremos las librerías necesarias y luego configuraremos:

- La conexión a PostgreSQL.  
- Los modelos de Ollama.  
- Las funciones auxiliares del pipeline RAG.  
- El sistema de chunking y embeddings.  

### Celda 1 — Instalación de dependencias
Ejecuta esta celda una sola vez para instalar todas las librerías necesarias del proyecto.


In [1]:
%pip install ollama psycopg2-binary pgvector tiktoken pymupdf python-dotenv -q



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Celda 2 — Configuración del entorno

En esta celda configuraremos:

- Variables de entorno.
- Conexión a PostgreSQL.
- Modelos utilizados por Ollama.
- Parámetros principales del sistema RAG.

También verificaremos que tanto PostgreSQL como Ollama estén funcionando correctamente antes de continuar.

Los parámetros de entorno serán los siguientes:
DB_HOST=localhost  
DB_PORT=5432  
DB_NAME=rag_demo  
DB_USER=postgres  
DB_PASSWORD=postgres  

---

In [ ]:
import os
import json
import psycopg2
import tiktoken
import ollama
import fitz  # pymupdf — extracción de texto desde PDFs

from pathlib import Path
from dotenv import load_dotenv
from psycopg2.extras import execute_values


# Cargar variables de entorno
load_dotenv()


# Configuración de PostgreSQL

# Con Docker Compose:
# DB_HOST=db

# En entorno local:
# DB_HOST=localhost

DB_CONFIG = {
    "host":     os.getenv("DB_HOST", "localhost"),
    "port":     int(os.getenv("DB_PORT", 5432)),
    "dbname":   os.getenv("DB_NAME", "rag_demo"),
    "user":     os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
}


# Configuración de modelos

# Modelo para embeddings
EMBED_MODEL = "nomic-embed-text"

# Modelo generativo
CHAT_MODEL = "llama3.2:1b"


# Parámetros del sistema RAG

CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
TOP_K = 5


def get_conn():
    """
    Retorna una conexión nueva a PostgreSQL.
    """
    return psycopg2.connect(**DB_CONFIG)



# Verificación de conexión

print("Verificando modelos y conexión...\n")

# Verificar embeddings
v = ollama.embed(
    model=EMBED_MODEL,
    input="test"
)["embeddings"][0]

print(f"Embeddings OK — dimensiones: {len(v)}")

# Verificar PostgreSQL
print(f"Conectando a PostgreSQL en {DB_CONFIG['host']}:{DB_CONFIG['port']}...")

with get_conn() as conn:
    print("PostgreSQL OK")

print("\nConfiguración lista")

### Celda 3 — Funciones auxiliares

Estas funciones serán utilizadas durante todo el pipeline RAG para:

- Generar embeddings.
- Procesar múltiples textos.
- Dividir documentos en chunks.

---

In [ ]:
def embed(text: str) -> list:
    """Convierte un texto en un vector de 768 números con nomic-embed-text."""
    return ollama.embed(model=EMBED_MODEL, input=text)["embeddings"][0]


def embed_batch(texts: list) -> list:
    """Vectoriza una lista de textos en una sola llamada (más eficiente que un loop)."""
    return ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"]


def chunk_text(text: str) -> list:
    """
    Divide el texto en fragmentos de CHUNK_SIZE tokens.
    Cada fragmento comparte CHUNK_OVERLAP tokens con el siguiente
    para no perder contexto en los bordes.
    """
    enc    = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = min(start + CHUNK_SIZE, len(tokens))
        chunks.append(enc.decode(tokens[start:end]).strip())
        start += CHUNK_SIZE - CHUNK_OVERLAP  # avanzar dejando overlap

    return chunks


# Verificar que Ollama responde antes de continuar
v = embed("test")
print(f"Ollama embed OK — dimensiones: {len(v)}")
print(f"Conectando a PostgreSQL en {DB_CONFIG['host']}:{DB_CONFIG['port']}...")
with get_conn() as conn:
    print("PostgreSQL OK ✓")
print("Configuración lista ✓")


Ollama embed OK — dimensiones: 768
Conectando a PostgreSQL en db:5432...
PostgreSQL OK ✓
Configuración lista ✓


---
### Celda 4 — Configuración de la base vectorial

En esta etapa prepararemos PostgreSQL para almacenar embeddings.

La función `setup_db()` realizará automáticamente:

- La activación de la extensión `pgvector`.
- La creación de la tabla `document_chunks`.
- La creación del índice vectorial HNSW para búsquedas semánticas rápidas.

---

### Estructura de la tabla

```text
document_chunks
│
├── id         → identificador único del chunk
├── source     → nombre del documento origen
├── chunk_idx  → posición del fragmento dentro del documento
├── content    → texto del chunk
└── embedding  → vector numérico del texto
```

---

### ¿Qué es un índice HNSW?

HNSW (*Hierarchical Navigable Small World*) es un tipo de índice optimizado para búsqueda vectorial.

Permite encontrar embeddings similares de manera mucho más rápida que una búsqueda secuencial tradicional.

En sistemas RAG esto es importante porque:
- La base puede contener miles o millones de fragmentos.
- Necesitamos recuperar resultados en milisegundos.
- El LLM depende de esa recuperación para responder correctamente.

---

In [3]:
def setup_db():
    """
    Crea la extensión vector, la tabla document_chunks y el índice HNSW.
    NOTA: hace DROP de la tabla si ya existe para asegurar el schema correcto.
    """
    with get_conn() as conn:
        with conn.cursor() as cur:

            # Habilitar pgvector (solo la primera vez por base de datos)
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

            # Recrear la tabla limpia — elimina la versión anterior si existe
            # (necesario si venías de una versión con chunk_type)
            cur.execute("DROP TABLE IF EXISTS document_chunks CASCADE;")

            cur.execute("""
                CREATE TABLE document_chunks (
                    id        SERIAL  PRIMARY KEY,
                    source    TEXT    NOT NULL,
                    chunk_idx INTEGER NOT NULL,
                    content   TEXT    NOT NULL,
                    embedding vector(768)
                );
            """)

            # HNSW para cosine similarity
            cur.execute("""
                CREATE INDEX chunks_hnsw_idx
                    ON document_chunks
                    USING hnsw (embedding vector_cosine_ops);
            """)

        conn.commit()
    print("DB lista — tabla document_chunks recreada + índice HNSW ✓")


setup_db()


DB lista — tabla document_chunks recreada + índice HNSW ✓


---
# 5. Indexing de texto

### Celda 5 — Pipeline de indexación del documento

En esta etapa construiremos el pipeline completo de ingesta del PDF.

El flujo realiza automáticamente:

1. **Extracción de texto**  
   PyMuPDF lee el documento página por página y concatena todo el contenido.

2. **Chunking**  
   El texto se divide en fragmentos de 400 tokens con overlap de 50 tokens para preservar contexto entre chunks consecutivos.

3. **Generación de embeddings**  
   Todos los chunks se vectorizan en batch utilizando `nomic-embed-text`.

4. **Almacenamiento en pgvector**  
   Los embeddings y fragmentos son insertados masivamente en PostgreSQL.


---

### Inserción idempotente

La indexación es idempotente.

Esto significa que:
- Si el documento ya existe en la base,
- Primero se eliminan sus registros anteriores,
- Y luego se inserta nuevamente la versión actualizada.

De esta manera evitamos duplicados durante las pruebas del notebook.

---


In [4]:
def index_pdf(pdf_path: str):
    """
    Indexa un PDF en pgvector:
    extrae texto → chunking → embed batch → inserción masiva.
    """
    source = Path(pdf_path).name
    print(f"Indexando: {source}")

    # Extraer texto de todas las páginas
    doc  = fitz.open(pdf_path)
    text = "\n".join(p.get_text() for p in doc)

    chunks = chunk_text(text)
    print(f"  → {len(chunks)} chunks generados")

    # Embed en batch: más rápido que llamar embed() en un loop
    embeddings = embed_batch(chunks)
    print(f"  → {len(embeddings)} embeddings generados")

    # Preparar filas para inserción masiva
    rows = [
        (source, idx, chunk, json.dumps(emb))
        for idx, (chunk, emb) in enumerate(zip(chunks, embeddings))
    ]

    with get_conn() as conn:
        with conn.cursor() as cur:
            # Borrar indexación previa del mismo archivo (idempotente)
            cur.execute("DELETE FROM document_chunks WHERE source=%s;", (source,))
            execute_values(
                cur,
                "INSERT INTO document_chunks (source, chunk_idx, content, embedding) VALUES %s",
                rows,
                template="(%s, %s, %s, %s::vector)",
            )
        conn.commit()

    print(f"  → {len(rows)} chunks guardados en pgvector ✓")


index_pdf("techpulse_ia_2025.pdf")


Indexando: techpulse_ia_2025.pdf
  → 6 chunks generados
  → 6 embeddings generados
  → 6 chunks guardados en pgvector ✓



# 6. Retrieval y busqueda semantica

### Celda 6 — Recuperación de chunks similares

Una vez indexados los documentos, el siguiente paso es recuperar los fragmentos más relevantes para una pregunta.

Para lograrlo:

1. La pregunta del usuario se convierte en un embedding.
2. pgvector compara ese embedding contra todos los embeddings almacenados.
3. Se recuperan los chunks más similares semánticamente.

---

### ¿Cómo mide similitud pgvector?

El operador `<=>` calcula la distancia coseno entre dos vectores.

```text
Menor distancia  →  Mayor similitud semántica
Mayor distancia  →  Menor similitud semántica
```

---

Usamos `1 - (embedding <=> query_vec)` para convertirlo en un **score de similitud**
entre 0 y 1, más intuitivo para mostrar en los resultados.

- `1.0` → muy similar
- `0.0` → poco relacionado


In [5]:
def retrieve(query: str, top_k: int = TOP_K) -> list:
    """
    Vectoriza la pregunta y recupera los top_k chunks más similares de pgvector.
    Retorna una lista de dicts con source, chunk_idx, content y similarity.
    """
    q_vec = json.dumps(embed(query))

    sql = """
        SELECT
            source,
            chunk_idx,
            content,
            1 - (embedding <=> %s::vector) AS similarity
        FROM document_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, [q_vec, q_vec, top_k])
            rows = cur.fetchall()

    return [
        {
            "source":    r[0],
            "chunk_idx": r[1],
            "content":   r[2],
            "similarity": round(float(r[3]), 4),
        }
        for r in rows
    ]


# Demo: ver qué chunks recupera una pregunta
query = "¿Qué herramientas usan los desarrolladores en LATAM?"
hits  = retrieve(query)

print(f"Query: {query}\n")
for h in hits:
    print(f"  [{h['similarity']:.4f}] {h['source']}")
    print(f"  {h['content'][:120].replace(chr(10), ' ')}...\n")


Query: ¿Qué herramientas usan los desarrolladores en LATAM?

  [0.7034] techpulse_ia_2025.pdf
  TechPulse Research  |  Reporte Q2 2025 Estado del Arte: IA Generativa en Ingenieria de Software 2025 TechPulse Research ...

  [0.6557] techpulse_ia_2025.pdf
  % interanual) - Cursor / Windsurf IDEs IA      -  18% Casos de uso principales - Autocompletado y generacion de codigo b...

  [0.6442] techpulse_ia_2025.pdf
  izan o eliminan facilmente. - Privacidad: con Ollama + nomic-embed, los datos nunca salen de la infraestructura. Stack t...

  [0.6318] techpulse_ia_2025.pdf
  ecosistema (pgvector, Ollama, LangChain) reduce la barrera de entrada a dias. Los equipos reportan ROI positivo en menos...

  [0.6063] techpulse_ia_2025.pdf
  ario, soporte para Llama 3, Mistral, Phi-3 y nomic-embed-text. Adopcion critica en fintech, salud y gobierno por privaci...



---
# 7. Generacion de respuestas con RAG
### Celda 7 — Prompt aumentado y generación con LLM

En esta etapa completaremos el pipeline RAG.

El flujo realiza:

1. Recuperar los chunks más relevantes desde pgvector.
2. Construir un contexto aumentado utilizando esos fragmentos.
3. Enviar el contexto al modelo LLM.
4. Generar una respuesta basada únicamente en la información recuperada.

---

### ¿Qué es un prompt aumentado?

Un prompt aumentado (*augmented prompt*) es un prompt al que se le agrega contexto externo antes de enviarlo al modelo.

En RAG, el contexto normalmente proviene de:
- Bases vectoriales.
- Documentos indexados.
- PDFs.
- Wikis o bases de conocimiento.

Esto permite que el modelo responda usando información específica que no estaba en su entrenamiento original.


### Temperatura del modelo

Utilizamos:

```python
temperature = 0.2
```

Una temperatura baja produce respuestas:
- Más precisas.
- Más consistentes.
- Más fieles al contexto recuperado.

Temperaturas más altas generan respuestas más creativas, pero aumentan el riesgo de alucinaciones.



In [6]:
SYSTEM_PROMPT = """Eres un asistente especializado en IA y desarrollo de software.
Responde ÚNICAMENTE usando los fragmentos de contexto proporcionados.
Cada fragmento indica su fuente.
Si la respuesta no está en el contexto, indícalo explícitamente.
Cita la fuente cuando sea relevante."""


def build_context(chunks: list) -> str:
    """Concatena los chunks en un bloque de contexto etiquetado por fuente."""
    parts = [
        f"[{i} | Fuente: {c['source']} | Similitud: {c['similarity']}]\n{c['content']}"
        for i, c in enumerate(chunks, 1)
    ]
    return "\n\n---\n\n".join(parts)


def ask(question: str, top_k: int = TOP_K) -> str:
    """
    Pipeline RAG completo (Retrieval → Augment → Generate):
      1. Vectoriza la pregunta con nomic-embed-text
      2. Recupera top_k chunks relevantes de pgvector
      3. Construye el prompt aumentado con el contexto
      4. llama3.1:8b genera la respuesta localmente
    """
    chunks   = retrieve(question, top_k)
    context  = build_context(chunks)
    user_msg = f"Contexto:\n{context}\n\nPregunta: {question}"

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        options={"temperature": 0.2},  # baja temperatura = respuestas precisas
    )
    return response["message"]["content"]


print("Pipeline RAG listo ✓")


Pipeline RAG listo ✓


---
# 8. Demo de preguntas
### Celda 8 — Preguntas sobre el reporte TechPulse

En esta última etapa ejecutaremos el pipeline RAG completo utilizando preguntas reales sobre el documento indexado.

Para cada consulta, el sistema realizará automáticamente:

1. Generación del embedding de la pregunta.
2. Recuperación de los chunks más relevantes desde pgvector.
3. Construcción del prompt aumentado.
4. Generación de la respuesta utilizando `llama3.2:1b`.

Corremos el pipeline completo con cinco preguntas.
El sistema recupera los chunks más relevantes y genera respuestas
contextualizadas.


In [7]:
preguntas = [
    "¿Qué porcentaje de equipos usa pgvector como vector store?",
    "¿Por qué RAG es preferido sobre fine-tuning para datos que cambian frecuentemente?",
    "¿Qué ventajas tiene usar Ollama con modelos locales?",
    "¿Cuántas horas semanales ahorra RAG por desarrollador?",
    "¿Qué es Agentic RAG y por qué es una tendencia emergente?",
]

for pregunta in preguntas:
    print(f"\n{'='*65}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*65}")
    print(ask(pregunta))



Pregunta: ¿Qué porcentaje de equipos usa pgvector como vector store?
Según el texto proporcionado, el porcentaje de equipos que usan pgvector como vector store es del 38%. Esto se refleja en la siguiente cita:

"Stack tecnologico mas comun para RAG (2025)
- Vector stores: pgvector/PostgreSQL 38% | Qdrant 22% | ChromaDB 19% | Pinecone 15%"

Pregunta: ¿Por qué RAG es preferido sobre fine-tuning para datos que cambian frecuentemente?
No se proporcionó una pregunta específica en el contexto proporcionado. Sin embargo, puedo ofrecerte una respuesta general basada en los fragmentos de contexto proporcionados.

Según la información proporcionada, RAG (Retrieval-Augmented Generation) es preferido sobre fine-tuning para datos que cambian frecuentemente debido a las siguientes razones:

1. **Costo**: Fine-tuning GPT-4 puede costar USD 50K a 2M, mientras que RAG utiliza el modelo base sin cambios.
2. **Velocidad**: RAG incorpora nuevos documentos en minutos, lo que la hace más rápida y eficiente

---
### Celda 9 — Inspección del estado de la base de datos

Útil para verificar cuántos chunks fueron indexados y con qué longitud promedio.


In [8]:
def db_stats():
    """Muestra un resumen de los chunks indexados por fuente."""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT
                    source,
                    COUNT(*)                  AS total_chunks,
                    AVG(LENGTH(content))::INT AS avg_chars
                FROM document_chunks
                GROUP BY source
                ORDER BY source;
            """)
            rows = cur.fetchall()

    print(f"{'Fuente':<35} {'Chunks':>8} {'Avg chars':>10}")
    print("-" * 55)
    for r in rows:
        print(f"{r[0]:<35} {r[1]:>8} {r[2]:>10}")


db_stats()


Fuente                                Chunks  Avg chars
-------------------------------------------------------
techpulse_ia_2025.pdf                      6       1135


---
### Celda 10 — Tu turno

Prueba tu propia pregunta sobre el documento indexado.
También puedes indexar otro PDF y comparar los resultados.


In [23]:
# Pregunta libre
mi_pregunta = "¿Cuáles son las principales amenazas de seguridad en sistemas RAG?"

print(f"Pregunta: {mi_pregunta}\n")
print(ask(mi_pregunta))


Pregunta: ¿Cuáles son las principales amenazas de seguridad en sistemas RAG?

La pregunta se refiere a las principales amenazas de seguridad en sistemas RAG (Retrieval-Augmented Generation). Según el texto proporcionado, algunas de las amenazas mencionadas incluyen:

* Alucinaciones: LLM mezcla contexto recuperado con información inventada
* Prompt injection en sistemas RAG con inputs no confiables
* Filtración de información confidencial a través de respuestas
* Dependencia de APIs externas y riesgo de discontinuación

Estas amenazas se mencionan como principales preocupaciones de seguridad para los sistemas RAG, lo que sugiere que la seguridad es un aspecto crítico en el desarrollo y implementación de estos sistemas.


## Hoy aprendimos

- A desplegar un entorno con Docker sin APIS externas y usando recursos locales  
- Los fundamentos de los sistemas RAG  
- La estructura básica de un sistema LLM  
  